# 1x1-conv-channel-reshape — worked example 3: Construct a 1x1 Conv2d that copies a Linear layer

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `1x1-conv-channel-reshape`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The 1x1-conv / per-pixel-Linear equivalence runs both directions. Given a trained `nn.Linear(C_in, C_out)`, you can build a 1x1 `nn.Conv2d` that produces identical per-pixel outputs by reshaping `linear.weight` from `(C_out, C_in)` into the conv weight shape `(C_out, C_in, 1, 1)` and copying the bias. This is how a fully-connected classifier head gets 'convolutionalized' into a fully-convolutional network.

## Worked solution

We start from a `Linear(5, 3)` and manufacture a matching `Conv2d(5, 3, 1)`.

1. **Make the conv shell.** `nn.Conv2d(C_in, C_out, kernel_size=1)` allocates a weight of shape `(C_out, C_in, 1, 1) = (3, 5, 1, 1)` and a bias `(3,)`. Its random init is irrelevant — we overwrite it.

2. **Reshape the Linear weight into conv layout.** `linear.weight` is `(C_out, C_in) = (3, 5)`. Add the two singleton kernel dims: `linear.weight.view(C_out, C_in, 1, 1)`. We copy with `.data` (and `.clone()`) so we do not entangle the two modules' autograd graphs.

3. **Copy the bias straight across.** Both biases are shape `(C_out,)`, so `conv.bias.data = linear.bias.data.clone()`.

4. **Verify on an image.** Run the conv on `(B, C_in, H, W)`. Independently, flatten the same input to pixels, push through the original linear, and fold back. The two outputs match to fp tolerance, confirming the conv is a faithful per-pixel copy of the linear.

This direction is the trick behind turning a `Linear` classifier head into a `1x1 Conv` so the network accepts arbitrary input sizes.

In [ ]:
import torch.nn as nn

def conv_from_linear(linear):
    OC, IC = linear.weight.shape
    conv = nn.Conv2d(IC, OC, kernel_size=1)
    conv.weight.data = linear.weight.data.view(OC, IC, 1, 1).clone()
    conv.bias.data = linear.bias.data.clone()
    return conv

t.manual_seed(0)
linear = nn.Linear(5, 3)
conv = conv_from_linear(linear)
x = t.randn(2, 5, 4, 6)
conv_out = conv(x)

# independent per-pixel linear reference
x_flat = rearrange(x, 'b c h w -> (b h w) c')
lin_out = rearrange(linear(x_flat), '(b h w) c -> b c h w', b=2, h=4, w=6)
print('conv output shape:', tuple(conv_out.shape))
print('matches per-pixel linear:', t.allclose(conv_out, lin_out, atol=1e-5))